# Crossgen Biosignature Model — POSEIDON Evaluation

Hybrid quantum-classical model trained on TauREx, tested on POSEIDON.
- **Train/Val**: TauREx (37,281 / 4,142 samples)
- **Test**: POSEIDON (685 samples)
- **Spectra**: Rebinned from 218 → 44 bins (Ariel wavelength grid)

## Load Trained Model

In [1]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from crossgen_hybrid_training import (
    TrainingConfig, prepare_data, build_model, evaluate_split,
    resolve_training_device, TARGET_COLS, move_prepared_data_to_device,
)

output_dir = Path("outputs/model_crossgen_rebinned")

config = TrainingConfig()
device = resolve_training_device(config)
data = prepare_data(config)
data = move_prepared_data_to_device(data, device)

model = build_model(config, device)
ckpt = torch.load(output_dir / "best_model.pt", weights_only=False, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
print(f"Loaded best model from epoch {ckpt['best_epoch']}, val_loss={ckpt['best_val_loss']:.5f}")

Rebinned spectra: 218 bins -> 44 bins (0.95 - 4.91 um)
Loaded best model from epoch 29, val_loss=0.29991


## Cross-Generator Evaluation: TauREx vs POSEIDON

In [2]:
val_metrics = evaluate_split(model, data.inner_val, data.target_scaler, config.eval_batch_size)
test_metrics = evaluate_split(model, data.test, data.target_scaler, config.eval_batch_size)

comparison = pd.DataFrame({
    "Target": TARGET_COLS,
    "Val RMSE (TauREx)": val_metrics["rmse_orig"],
    "Test RMSE (POSEIDON)": test_metrics["rmse_orig"],
})
comparison["Gap"] = comparison["Test RMSE (POSEIDON)"] - comparison["Val RMSE (TauREx)"]
comparison.loc[len(comparison)] = [
    "MEAN",
    val_metrics["rmse_mean"],
    test_metrics["rmse_mean"],
    test_metrics["rmse_mean"] - val_metrics["rmse_mean"],
]

print("Cross-Generator Generalization: TauREx (train) \u2192 POSEIDON (test)")
print("=" * 70)
print(comparison.to_string(index=False, float_format="%.4f"))

Cross-Generator Generalization: TauREx (train) → POSEIDON (test)
       Target  Val RMSE (TauREx)  Test RMSE (POSEIDON)    Gap
log10_vmr_h2o             1.6620                3.1307 1.4687
log10_vmr_co2             1.1581                4.7260 3.5680
 log10_vmr_co             2.1817                3.2832 1.1015
log10_vmr_ch4             1.2188                3.2275 2.0087
log10_vmr_nh3             1.5032                2.9483 1.4451
         MEAN             1.5448                3.4631 1.9184


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

targets_short = [t.replace("log10_vmr_", "") for t in TARGET_COLS]
x = np.arange(len(TARGET_COLS))
width = 0.35

ax = axes[0]
ax.bar(x - width/2, val_metrics["rmse_orig"], width, label="Val (TauREx)", color="#4C72B0")
ax.bar(x + width/2, test_metrics["rmse_orig"], width, label="Test (POSEIDON)", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(targets_short)
ax.set_ylabel("RMSE (log10 VMR)")
ax.set_title("Per-Target RMSE: TauREx vs POSEIDON")
ax.legend()

ax = axes[1]
history = pd.read_csv(output_dir / "history.csv")
ax.plot(history["epoch"], history["inner_val_rmse_mean"], label="Val RMSE (TauREx)", color="#4C72B0")
ax.axhline(y=test_metrics["rmse_mean"], color="#DD8452", linestyle="--", label=f"Test RMSE (POSEIDON): {test_metrics['rmse_mean']:.2f}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean RMSE")
ax.set_title("Training Progress vs POSEIDON Test")
ax.legend()

plt.tight_layout()
plt.savefig(output_dir / "crossgen_evaluation.png", dpi=150)
plt.show()

/var/folders/wh/9tfsnwk52msftggvvzkn8gw00000gn/T/ipykernel_68507/2493138276.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Training (optional — skip if model already trained)

**WARNING: Running the cell below will retrain from scratch and overwrite saved weights.**

In [ ]:
# UNCOMMENT TO RETRAIN:
# from crossgen_hybrid_training import TrainingConfig, run_training_experiment
# result = run_training_experiment(TrainingConfig())